# Document Graph: Access Control & Security Trimming

This notebook demonstrates the end-to-end **Security Trimming and Access Control** architecture in `genai-graph`:

1. **Security Trimming Principles**:
   - Early-binding (pre-filtering) vs. late-binding (post-filtering).
   - Principal expansion (`user:<id>`, `group:<id>`, `role:<name>`, `public`).
   - Preventing Top-$K$ vector starvation and side-channel timing leaks.
2. **Setting Up Corpus & ACL Provider**:
   - Establishing multi-tier directory structure (`public/`, `finance/`, `hr/`, `executive/`).
   - Defining path matching rules in `YamlAccessControlProvider`.
3. **Build-Time Ingestion & Inheritance**:
   - Ingesting into Ladybug DB via `DocumentGraphFactory` and `ingest_document_graph`.
   - Inspecting `allowed_principals` stored on `Folder` and `Document` nodes.
4. **Runtime User Context Propagation**:
   - Constructing `UserContext` for different organizational roles.
   - Dual-propagation via LangGraph `ToolRuntime` and thread/task `contextvars`.
5. **Interactive Tool Navigation & Query Trimming**:
   - Security-trimmed `list_documents` with configurable feedback notices.
   - Folder (`get_folder_toc`) and Document TOC (`get_document_toc`) access validation.
   - Section reading (`get_section_content`) and search (`search_sections`) pre-filtering.
6. **Enterprise Extension (SharePoint & Microsoft Graph)**:
   - Blueprint for Microsoft 365, Delta Queries, and Entra ID integration.

## 1. Environment & Setup

We set up a temporary directory for our document corpus, access control rules, and a dedicated Ladybug graph database.

In [ ]:
import tempfile
from pathlib import Path

import yaml

# Create isolated working directory
WORK_DIR = Path(tempfile.mkdtemp(prefix="docgraph_acl_"))
CORPUS_DIR = WORK_DIR / "corpus"
DB_PATH = WORK_DIR / "docgraph_acl.db"
RULES_FILE = WORK_DIR / "access_rules.yaml"

print(f"Working Directory : {WORK_DIR}")
print(f"Corpus Directory  : {CORPUS_DIR}")
print(f"Database Path     : {DB_PATH}")

## 2. Setting Up Sample Corpus & Access Control Rules

We create four distinct document directories reflecting enterprise confidentiality boundaries:
- `public/`: General company overview and product documentation.
- `finance/`: Quarterly financial reports and budgets.
- `hr/confidential/`: Employee compensation and payroll reviews.
- `executive/`: Board meeting minutes and strategic acquisitions.

In [ ]:
# Create directory structure
(CORPUS_DIR / "public").mkdir(parents=True, exist_ok=True)
(CORPUS_DIR / "finance").mkdir(parents=True, exist_ok=True)
(CORPUS_DIR / "hr" / "confidential").mkdir(parents=True, exist_ok=True)
(CORPUS_DIR / "executive").mkdir(parents=True, exist_ok=True)

# 1. Public document
(CORPUS_DIR / "public" / "company_handbook.md").write_text(
    """# Company Handbook

Welcome to Atos GenAI Platform.
Our core mission is delivering trusted, high-performance enterprise AI solutions.

## General Information
Office hours are 9:00 AM to 6:00 PM. Remote work guidelines apply to all team members.
""",
    encoding="utf-8",
)

# 2. Finance document
(CORPUS_DIR / "finance" / "q3_revenue.md").write_text(
    """# Q3 Financial Report

Confidential financial performance and forecasts for Q3 2026.

## Revenue Breakdown
- Cloud Solutions: $45.2M (+14% YoY)
- Enterprise AI Services: $28.6M (+38% YoY)
- Legacy Maintenance: $12.1M (-8% YoY)

## Operating Margin
Total operating margin reached 22.4% this quarter.
""",
    encoding="utf-8",
)

# 3. HR Confidential document
(CORPUS_DIR / "hr" / "confidential" / "payroll_summary.md").write_text(
    """# 2026 Payroll & Compensation Review

Strictly confidential HR data regarding compensation bands and bonus allocations.

## Compensation Bands
- Senior AI Engineer: $160,000 - $210,000 + 15% bonus target
- Principal Architect: $210,000 - $265,000 + 20% bonus target

## Retention Equity Pool
Executive leadership approved an additional $2.5M retention stock grant pool.
""",
    encoding="utf-8",
)

# 4. Executive document
(CORPUS_DIR / "executive" / "board_minutes_2026.md").write_text(
    """# Executive Board Strategy

Top secret board minutes regarding corporate restructuring and M&A pipeline.

## Project Falcon Acquisition
Negotiations to acquire CloudGraph Systems for $140M are entering Phase 3 due diligence.
Target closing date: Q1 2027.
""",
    encoding="utf-8",
)

# Create Access Control Rules YAML
rules_config = {
    "default": ["public"],
    "rules": [
        {
            "path": "public/**",
            "allowed_principals": ["public"],
            "inheritance": "override",
        },
        {
            "path": "finance/**",
            "allowed_principals": ["group:finance_dept", "role:cfo"],
            "inheritance": "override",
        },
        {
            "path": "hr/confidential/**",
            "allowed_principals": ["group:hr_leadership", "user:alice"],
            "inheritance": "intersection",
        },
        {
            "path": "executive/**",
            "allowed_principals": ["group:board_members", "role:executive"],
            "inheritance": "override",
        },
    ],
}

with open(RULES_FILE, "w", encoding="utf-8") as f:
    yaml.safe_dump(rules_config, f)

print(f"Created 4 documents and rules file at {RULES_FILE}")

## 3. Build-Time Ingestion with `YamlAccessControlProvider`

We instantiate the `YamlAccessControlProvider` and build the Document Graph using `DocumentGraphFactory` and `ingest_document_graph`.

The provider resolves file paths against glob patterns, tagging every `Folder` and `Document` node with its effective `allowed_principals`.

In [ ]:
from genai_graph.kg.access.yaml_provider import YamlAccessControlProvider
from genai_graph.kg.backend import KuzuBackend
from genai_graph.kg.document_graph.ingest import ingest_document_graph
from genai_graph.kg.factories.document_graph_factory import DocumentGraphFactory

# 1. Initialize ACL provider with YAML rules
acl_provider = YamlAccessControlProvider(yaml_path=RULES_FILE)

# 2. Build DocumentGraphFactory with ACL resolution
factory = DocumentGraphFactory(
    sources=[str(CORPUS_DIR)],
    access_control_provider=acl_provider,
    inheritance_mode="override",
)

# 3. Connect to Ladybug DB and ingest
backend = KuzuBackend()
backend.connect(str(DB_PATH))

ingest_stats = ingest_document_graph(backend, factory)
print("Ingestion complete!")
print(f"Documents processed: {ingest_stats.documents_processed}")
print(f"Total sections     : {ingest_stats.sections_created}")

### Inspecting Ingested Document & Folder Nodes

Let's run a Cypher query to verify that `allowed_principals` was correctly stored across all documents in the database.

In [ ]:
df_docs = backend.execute_get_as_df(
    "MATCH (d:Document) RETURN d.filename AS filename, d.allowed_principals AS allowed_principals, d.token_count AS tokens ORDER BY d.filename",
    None,
)
df_docs

## 4. Runtime User Context & Security Principals

We define test personas using `UserContext`.

When a request arrives at runtime:
1. The user's JWT/token is expanded into effective security principals (`user:<id>`, `group:<id>`, `role:<name>`, `public`).
2. The `UserContext` is propagated down through the agent harness to the graph tools.

In [ ]:
from genai_graph.kg.access.context import UserContext, set_active_user_context

# Persona 1: Anonymous / Public User
public_user = UserContext.create()

# Persona 2: Finance Analyst (Bob)
finance_user = UserContext.create(
    user_id="bob",
    groups=["finance_dept", "all_employees"],
    roles=["analyst"],
)

# Persona 3: HR Lead (Alice)
hr_user = UserContext.create(
    user_id="alice",
    groups=["hr_leadership", "all_employees"],
    roles=["hr_partner"],
)

# Persona 4: Board Member & Executive (Carol)
exec_user = UserContext.create(
    user_id="carol",
    groups=["board_members"],
    roles=["executive"],
)

# Persona 5: System Admin (Superuser)
admin_user = UserContext.create(
    user_id="root",
    is_admin=True,
)

print(f"Public User Principals : {public_user.principals}")
print(f"Finance User Principals: {finance_user.principals}")
print(f"HR User Principals     : {hr_user.principals}")
print(f"Executive Principals   : {exec_user.principals}")
print(f"Admin Bypass           : {admin_user.is_admin}")

## 5. Security-Trimmed Navigation & Discovery

We initialize the Document Graph tools with `trimming_feedback="aggregate_notice"`.

Notice how:
- `list_documents` returns **only** documents accessible to the active persona.
- An aggregate note informs the agent how many documents were omitted without disclosing confidential titles.

In [ ]:
from genai_graph.kg.query.document_graph_tools import create_document_graph_tools

tools = {t.name: t for t in create_document_graph_tools(str(DB_PATH), trimming_feedback="aggregate_notice")}

# 1. Listing documents as Public User
set_active_user_context(public_user)
print("--- [Public User View] ---")
print(tools["list_documents"].invoke({}))

# 2. Listing documents as Finance User (Bob)
set_active_user_context(finance_user)
print("\n--- [Finance Analyst View] ---")
print(tools["list_documents"].invoke({}))

# 3. Listing documents as Executive (Carol)
set_active_user_context(exec_user)
print("\n--- [Executive View] ---")
print(tools["list_documents"].invoke({}))

### 5.1 Document Table of Contents & Direct Access Control

If a user tries to access the table of contents or content of a restricted document directly (e.g. `get_document_toc` or `get_section_content`), the tool enforces security trimming and returns an explicit `Access Denied` error.

In [ ]:
# Attempting to fetch Executive Board document TOC as Public User
set_active_user_context(public_user)
print("Public User fetching 'board_minutes_2026.md':")
print(tools["get_document_toc"].invoke({"document_id": "board_minutes_2026.md"}))

# Authorized fetch by Executive Carol
set_active_user_context(exec_user)
print("\nExecutive Carol fetching 'board_minutes_2026.md':")
print(tools["get_document_toc"].invoke({"document_id": "board_minutes_2026.md"}))

### 5.2 Section Reading & Content Isolation

Let's read the section content for `payroll_summary.md`.
- Public user is denied access.
- HR lead Alice is granted access.

In [ ]:
# Public User attempting to read payroll
set_active_user_context(public_user)
payroll_res_pub = tools["get_section_content"].invoke({"section_ids": "payroll_summary.md"})
print(f"Public User result:\n{payroll_res_pub}")

# HR Alice reading payroll
set_active_user_context(hr_user)
payroll_res_alice = tools["get_section_content"].invoke({"section_ids": "payroll_summary.md"})
print(f"\nHR Alice result:\n{payroll_res_alice}")

### 5.3 Early-Binding Search Pre-Filtering

When searching across sections via `search_sections`, security trimming evaluates the user's permitted documents **before** ranking and scoring.
This guarantees zero information leakage and prevents Top-$K$ starvation.

In [ ]:
# Search for financial term 'revenue' as Public User
set_active_user_context(public_user)
search_pub = tools["search_sections"].invoke({"query": "revenue", "mode": "cypher"})
print(f"Public User search for 'revenue':\n{search_pub}")

# Search for financial term 'revenue' as Finance User (Bob)
set_active_user_context(finance_user)
search_fin = tools["search_sections"].invoke({"query": "revenue", "mode": "cypher"})
print(f"\nFinance Bob search for 'revenue':\n{search_fin}")

## 6. Extending to SharePoint & Enterprise Connectors

### SharePoint / Microsoft 365 Architecture

To extend DocGraph to SharePoint Online libraries:
1. **Build-Time Extraction**:
   - Query Microsoft Graph API (`GET /drives/{drive-id}/items/{item-id}/permissions`).
   - Extract `granted_to_v2` User Object IDs and Security Group SIDs.
   - Map SharePoint groups and Entra ID groups to `group:<object-id>` principals.
2. **Incremental Delta Synchronization**:
   - Track additions, deletions, and ACL modifications via Delta queries (`/drives/{id}/root/delta`).
3. **Runtime Identity Resolution**:
   - Expand user OAuth tokens via `GET /v1.0/me/transitiveMemberOf` to resolve all group memberships into `UserContext`.

```
┌────────────────────────────────────────────────────────┐
│               Microsoft Entra ID / IdP                 │
│         User SIDs + Transitive Group Memberships       │
└───────────────────────────┬────────────────────────────┘
                            │
               ┌────────────▼────────────┐
               │    UserContext Object   │
               │ principals: {"group:.."}│
               └────────────┬────────────┘
                            │
               ┌────────────▼────────────┐
               │   DocGraph Query Engine │
               │ Early-Binding Pre-Filter│
               └────────────┬────────────┘
                            │
               ┌────────────▼────────────┐
               │  Ladybug Database Index │
               │ Doc.allowed_principals  │
               └─────────────────────────┘
```

In [ ]:
# Clean up temporary test files
import shutil

shutil.rmtree(WORK_DIR, ignore_errors=True)
print("Demo completed successfully and temp files cleaned up.")